<a href="https://colab.research.google.com/github/Khushibung05/RAG/blob/main/chunking2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#Step 0: Create Employee Policy PDF
!pip install reportlab
from reportlab.pdfgen import canvas

pdf_path = "employee_policy.pdf"

c = canvas.Canvas(pdf_path)

content = """
Leave Policy

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.

Work From Home Policy

Employees may work from home twice per week.
Manager approval is required for additional remote work.

Travel Policy

Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.

Medical Insurance Policy

All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
"""

y = 800

for line in content.split("\n"):
    c.drawString(50, y, line)
    y -= 20

c.save()

print("PDF Created Successfully!")
print("Saved as:", pdf_path)
#Verify PDF
import os

print("Exists:", os.path.exists("employee_policy.pdf"))
print("Size:", os.path.getsize("employee_policy.pdf"), "bytes")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 31.5 MB/s eta 0:00:00
PDF Created Successfully!
Saved as: employee_policy.pdf
Exists: True
Size: 1798 bytes


In [4]:
!pip install PyPDF2
from PyPDF2 import PdfReader

reader = PdfReader("employee_policy.pdf")

text = ""

for page in reader.pages:
    text += page.extract_text()

print("===== ORIGINAL DOCUMENT =====\n")
print(text)

print("\n===== DOCUMENT STATISTICS =====")
print("Total Characters:", len(text))
print("Total Words:", len(text.split()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.7 MB/s eta 0:00:00
===== ORIGINAL DOCUMENT =====

Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.


===== DOCUMENT STATISTICS =====
Total Characters: 530
Total Words: 76


In [5]:
#Task 2: Fixed Size Chunking
from textwrap import wrap

chunk_sizes = [100, 200, 300]

for size in chunk_sizes:

    print(f"\n{'='*60}")
    print(f"FIXED SIZE CHUNKING (Chunk Size={size})")
    print(f"{'='*60}")

    chunks = wrap(text, size)

    for i, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {i}")
        print("-"*40)
        print(chunk)
        print("Length:", len(chunk))
#Analysis
print("""
ANALYSIS

1. Sentences may be split in the middle.
2. Context can be lost across chunk boundaries.
3. Easy to implement.
4. Poor retrieval quality for long documents.
""")


FIXED SIZE CHUNKING (Chunk Size=100)

Chunk 1
----------------------------------------
Leave Policy Employees receive 12 casual leaves annually. Employees receive 15 sick leaves annually.
Length: 100

Chunk 2
----------------------------------------
Unused casual leaves cannot be carried forward. Work From Home Policy Employees may work from home
Length: 98

Chunk 3
----------------------------------------
twice per week. Manager approval is required for additional remote work. Travel Policy Travel
Length: 93

Chunk 4
----------------------------------------
expenses are reimbursed within 30 days. Original receipts must be submitted for reimbursement.
Length: 94

Chunk 5
----------------------------------------
Medical Insurance Policy All employees are covered under company medical insurance. Dependent
Length: 93

Chunk 6
----------------------------------------
coverage is available for spouse and children.
Length: 46

FIXED SIZE CHUNKING (Chunk Size=200)

Chunk 1
------------------

In [7]:
#Task 3: Recursive Chunking
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

configs = [
    (200,20),
    (200,50),
    (300,50)
]

for chunk_size, overlap in configs:

    print(f"\n{'='*60}")
    print(f"Chunk Size={chunk_size}, Overlap={overlap}")
    print(f"{'='*60}")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )

    chunks = splitter.split_text(text)

    for i, chunk in enumerate(chunks, start=1):
        print(f"\nChunk {i}")
        print("-"*40)
        print(chunk)
#Analysis
print("""
RECURSIVE CHUNKING ANALYSIS

Overlap preserves context between chunks.

Example:
Chunk 1 ends with:
'Employees may work from home'

Chunk 2 starts with:
'work from home twice per week'

This helps retrievers avoid losing important information.
""")


Chunk Size=200, Overlap=20

Chunk 1
----------------------------------------
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy

Chunk 2
----------------------------------------
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel Policy
Travel expenses are reimbursed within 30 days.

Chunk 3
----------------------------------------
Original receipts must be submitted for reimbursement.
Medical Insurance Policy
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.

Chunk Size=200, Overlap=50

Chunk 1
----------------------------------------
Leave Policy
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home Policy

Chunk 2
----------------------------------

In [10]:
#Task 4: Sentence-Based Chunking
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt_tab')

sentences = sent_tokenize(text)

print("TOTAL SENTENCES:", len(sentences))

for i, sentence in enumerate(sentences, start=1):
    print(f"\nSentence {i}")
    print("-"*40)
    print(sentence)
#Analysis
print("""
SENTENCE CHUNKING ANALYSIS

1. Sentences remain intact.
2. Meaning is preserved.
3. Better than fixed-size chunking.
4. Context spanning multiple sentences may still be lost.
""")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


TOTAL SENTENCES: 9

Sentence 1
----------------------------------------
Leave Policy
Employees receive 12 casual leaves annually.

Sentence 2
----------------------------------------
Employees receive 15 sick leaves annually.

Sentence 3
----------------------------------------
Unused casual leaves cannot be carried forward.

Sentence 4
----------------------------------------
Work From Home Policy
Employees may work from home twice per week.

Sentence 5
----------------------------------------
Manager approval is required for additional remote work.

Sentence 6
----------------------------------------
Travel Policy
Travel expenses are reimbursed within 30 days.

Sentence 7
----------------------------------------
Original receipts must be submitted for reimbursement.

Sentence 8
----------------------------------------
Medical Insurance Policy
All employees are covered under company medical insurance.

Sentence 9
----------------------------------------
Dependent coverage is available

In [11]:
#Task 5: Semantic Chunking

#Since policies are already section-wise.

sections = text.split("Policy")

semantic_chunks = []

for section in sections:
    section = section.strip()

    if section:
        semantic_chunks.append(section)

for i, chunk in enumerate(semantic_chunks, start=1):
    print(f"\nSemantic Chunk {i}")
    print("-"*50)
    print(chunk)
#Better Manual Semantic Chunking
semantic_chunks = {

"Leave Policy":
"""
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
""",

"Work From Home Policy":
"""
Employees may work from home twice per week.
Manager approval is required for additional remote work.
""",

"Travel Policy":
"""
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
""",

"Medical Insurance Policy":
"""
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.
"""
}

for title, content in semantic_chunks.items():
    print(f"\n{title}")
    print("-"*50)
    print(content)
#Analysis
print("""
SEMANTIC CHUNKING ANALYSIS

1. Groups related information together.
2. Highest contextual coherence.
3. Better retrieval quality.
4. Preferred for enterprise RAG systems.
""")


Semantic Chunk 1
--------------------------------------------------
Leave

Semantic Chunk 2
--------------------------------------------------
Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.
Work From Home

Semantic Chunk 3
--------------------------------------------------
Employees may work from home twice per week.
Manager approval is required for additional remote work.
Travel

Semantic Chunk 4
--------------------------------------------------
Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.
Medical Insurance

Semantic Chunk 5
--------------------------------------------------
All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.

Leave Policy
--------------------------------------------------

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.

In [12]:
import pandas as pd

comparison = pd.DataFrame({

"Chunking Method":[
    "Fixed Size",
    "Recursive",
    "Sentence",
    "Semantic"
],

"Context Preservation":[
    "Low",
    "High",
    "Medium",
    "Very High"
],

"Retrieval Quality":[
    "Low",
    "High",
    "Medium",
    "Very High"
],

"Implementation Complexity":[
    "Easy",
    "Moderate",
    "Easy",
    "High"
]

})

print(comparison)

  Chunking Method Context Preservation Retrieval Quality  \
0      Fixed Size                  Low               Low   
1       Recursive                 High              High   
2        Sentence               Medium            Medium   
3        Semantic            Very High         Very High   

  Implementation Complexity  
0                      Easy  
1                  Moderate  
2                      Easy  
3                      High  


In [13]:
#Task 7: Retrieval Simulation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

queries = [
    "How many casual leaves are provided?",
    "Can employees work from home?",
    "What is the travel reimbursement process?",
    "Who is covered under medical insurance?"
]

# Example using semantic chunks

chunks = list(semantic_chunks.values())

vectorizer = TfidfVectorizer()

chunk_vectors = vectorizer.fit_transform(chunks)

for query in queries:

    query_vector = vectorizer.transform([query])

    similarity = cosine_similarity(
        query_vector,
        chunk_vectors
    )

    best_match = similarity.argmax()

    print(f"\nQuery: {query}")
    print("\nRetrieved Chunk:")
    print(chunks[best_match])

    print("\nChunking Method: Semantic")
    print("="*60)


Query: How many casual leaves are provided?

Retrieved Chunk:

Employees receive 12 casual leaves annually.
Employees receive 15 sick leaves annually.
Unused casual leaves cannot be carried forward.


Chunking Method: Semantic

Query: Can employees work from home?

Retrieved Chunk:

Employees may work from home twice per week.
Manager approval is required for additional remote work.


Chunking Method: Semantic

Query: What is the travel reimbursement process?

Retrieved Chunk:

Travel expenses are reimbursed within 30 days.
Original receipts must be submitted for reimbursement.


Chunking Method: Semantic

Query: Who is covered under medical insurance?

Retrieved Chunk:

All employees are covered under company medical insurance.
Dependent coverage is available for spouse and children.


Chunking Method: Semantic


In [15]:
#Task 8: Recommendation Report
print("""
==================================================
RECOMMENDATION REPORT
==================================================

1. HR POLICY ASSISTANT
Recommended: Semantic Chunking

Reason:
Policies are naturally divided into sections.
Retrieval accuracy is very high.

--------------------------------------------------

2. LEGAL DOCUMENT ASSISTANT
Recommended: Recursive Chunking

Reason:
Legal clauses are interconnected.
Overlap preserves clause references.

--------------------------------------------------

3. MEDICAL DOCUMENT ASSISTANT
Recommended: Semantic Chunking

Reason:
Medical topics should remain grouped.
Maintains clinical context.

--------------------------------------------------

4. RESEARCH PAPER ASSISTANT
Recommended: Recursive Chunking

Reason:
Research concepts span multiple paragraphs.
Overlap improves contextual continuity.

--------------------------------------------------

FINAL CONCLUSION

Semantic Chunking:
Best Retrieval Quality

Recursive Chunking:
Best General-Purpose RAG Strategy

Enterprise Systems:
Semantic + Recursive Hybrid Approach
is considered the most effective.
==================================================
""")


RECOMMENDATION REPORT

1. HR POLICY ASSISTANT
Recommended: Semantic Chunking

Reason:
Policies are naturally divided into sections.
Retrieval accuracy is very high.

--------------------------------------------------

2. LEGAL DOCUMENT ASSISTANT
Recommended: Recursive Chunking

Reason:
Legal clauses are interconnected.
Overlap preserves clause references.

--------------------------------------------------

3. MEDICAL DOCUMENT ASSISTANT
Recommended: Semantic Chunking

Reason:
Medical topics should remain grouped.
Maintains clinical context.

--------------------------------------------------

4. RESEARCH PAPER ASSISTANT
Recommended: Recursive Chunking

Reason:
Research concepts span multiple paragraphs.
Overlap improves contextual continuity.

--------------------------------------------------

FINAL CONCLUSION

Semantic Chunking:
Best Retrieval Quality

Recursive Chunking:
Best General-Purpose RAG Strategy

Enterprise Systems:
Semantic + Recursive Hybrid Approach
is considered the mo